# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [16]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

In [17]:
from datasets import load_dataset
import pandas as pd

cols_needed = [ 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks',  'gsc_avg_position', 'gsc_data_available']

ds_train = load_dataset("FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet",
    split="train", token=hf_token)
df_train_source = ds_train.select_columns(cols_needed).to_pandas()
del ds_train

cols_needed = [ 'content_hash_id', 'word_count', 'is_published', 'content_updated_date']
ds_content = load_dataset("FlyRank/internship-warehouse",
    data_files="dim_content.parquet", split="train", token=hf_token)
df_content = ds_content.select_columns(cols_needed).to_pandas()
del ds_content

df_train_source = df_train_source.merge(df_content, on="content_hash_id", how="left", validate="m:1")


df_train_source = df_train_source[df_train_source['gsc_data_available'] == True]


In [18]:
df_train_source['report_date'] = pd.to_datetime(df_train_source['report_date'])
df_train_source['content_updated_date'] = pd.to_datetime(df_train_source['content_updated_date'])

df_train_source['word_count_is_stale_safe'] = (
    df_train_source['content_updated_date'].isna() |
    (df_train_source['content_updated_date'] <= df_train_source['report_date'])
)

In [19]:
import numpy as np
#Removing bad data, gsc_avg_position at 0, because most have near 0 impressions and clicks, indicating bad data for high position
#it should start at 1
df_train_source.loc[df_train_source['gsc_avg_position'] == 0, 'gsc_avg_position'] = np.nan

df_train = df_train_source[df_train_source['gsc_avg_position'].notna()].copy()


df_train["CTR"] = (df_train["gsc_clicks"] / df_train["gsc_impressions"]) * 100



df_train.sort_values(by=['content_hash_id', 'report_date'], ascending=True, inplace=True)


df_train['report_date'] = pd.to_datetime(df_train['report_date'])


pos_diff = df_train.groupby('content_hash_id')['gsc_avg_position'].diff()
days_diff = df_train.groupby('content_hash_id')['report_date'].diff().dt.days
df_train['trend_direction'] = (pos_diff / days_diff).fillna(0)


df_train.loc[~df_train['word_count_is_stale_safe'], 'word_count'] = np.nan

In [20]:
df_train['pos_bucket'] = pd.cut(df_train['gsc_avg_position'], bins=[0,3,10,20,100,500])

In [21]:
import numpy as np

df_train['word_count'] = df_train.groupby('pos_bucket', observed=True)['word_count'].transform(
    lambda x: x.fillna(x.median())
)

for col in ['gsc_impressions', 'gsc_avg_position', 'word_count', 'CTR']:
    df_train[col] = np.log1p(df_train[col])

for col in ['CTR','word_count', 'gsc_impressions', 'gsc_clicks']:
    train_median = df_train[col].median()
    df_train[col] = df_train[col].fillna(train_median)
df_train['trend_direction'] = np.sign(df_train['trend_direction']) * np.log1p(np.abs(df_train['trend_direction']))


feature_cols = ["gsc_impressions", "gsc_clicks", "word_count", "CTR", "trend_direction"]

In [22]:
import numpy as np
from datasets import load_dataset
ds_test_raw = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-06/data_0.parquet",
    split="train", token=hf_token
)

cols_needed = [ 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks',  'gsc_avg_position', 'gsc_data_available']

df_test_full = ds_test_raw.select_columns(cols_needed).to_pandas()
del ds_test_raw

rng = np.random.RandomState(42)
unique_pages = df_test_full['content_hash_id'].unique()
n_pages_target = 5000
sampled_pages = rng.choice(unique_pages, size=min(n_pages_target, len(unique_pages)), replace=False)

df_test_source_honest = df_test_full[df_test_full['content_hash_id'].isin(sampled_pages)].copy()
del df_test_full

df_test_source_honest = df_test_source_honest.merge(df_content, on="content_hash_id", how="left", validate="m:1")
df_test_source_honest = df_test_source_honest[df_test_source_honest['gsc_data_available'] == True]


In [23]:

df_test_source_honest['report_date'] = pd.to_datetime(df_test_source_honest['report_date'])
df_test_source_honest['content_updated_date'] = pd.to_datetime(df_test_source_honest['content_updated_date'])


df_test_source_honest['word_count_is_stale_safe'] = (
    df_test_source_honest['content_updated_date'].isna() |
    (df_test_source_honest['content_updated_date'] <= df_test_source_honest['report_date'])
)
print(df_test_source_honest['word_count_is_stale_safe'].value_counts(normalize=True))

word_count_is_stale_safe
True     0.68382
False    0.31618
Name: proportion, dtype: float64


In [24]:
df_test_source_honest.loc[df_test_source_honest['gsc_avg_position'] == 0, 'gsc_avg_position'] = np.nan
df_test_honest = df_test_source_honest[df_test_source_honest['gsc_avg_position'].notna()].copy()

df_test_honest["CTR"] = (df_test_honest["gsc_clicks"] / df_test_honest["gsc_impressions"]) * 100
df_test_honest.sort_values(by=['content_hash_id', 'report_date'], ascending=True, inplace=True)
df_test_honest['report_date'] = pd.to_datetime(df_test_honest['report_date'])

pos_diff = df_test_honest.groupby('content_hash_id')['gsc_avg_position'].diff()
days_diff = df_test_honest.groupby('content_hash_id')['report_date'].diff().dt.days
df_test_honest['trend_direction'] = (pos_diff / days_diff).fillna(0)

df_test_honest['pos_bucket'] = pd.cut(df_test_honest['gsc_avg_position'], bins=[0,3,10,20,100,500])
df_test_honest.loc[~df_test_honest['word_count_is_stale_safe'], 'word_count'] = np.nan
df_test_honest['word_count'] = df_test_honest.groupby('pos_bucket', observed=True)['word_count'].transform(
    lambda x: x.fillna(x.median())
)

for col in ['gsc_impressions', 'gsc_avg_position', 'word_count', 'CTR']:
    df_test_honest[col] = np.log1p(df_test_honest[col])

for col in ['CTR', 'word_count', 'gsc_impressions', 'gsc_clicks']:
    df_test_honest[col] = df_test_honest[col].fillna(train_median)

df_test_honest['trend_direction'] = np.sign(df_test_honest['trend_direction']) * np.log1p(np.abs(df_test_honest['trend_direction']))

df_test_honest = df_test_honest[(df_test_honest['is_published'] == True) & (df_test_honest['gsc_avg_position'] <= np.log1p(100))]


In [25]:
from sklearn.preprocessing import QuantileTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd

feature_cols = ["gsc_impressions", "word_count", "CTR", "trend_direction", "gsc_avg_position"]

df_train_k = df_train[feature_cols]


scaler = QuantileTransformer(output_distribution='normal', random_state=42)
X_train_scaled = scaler.fit_transform(df_train_k).astype(np.float32)



pca = PCA(n_components=3,random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)


rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_train_pca), size=min(1_000_000, len(X_train_pca)), replace=False)
X_sample = X_train_pca[sample_idx]



km_final = KMeans(n_clusters=3, init='k-means++', n_init=20, random_state=42, max_iter=300)
labels = km_final.fit_predict(X_sample)
score = silhouette_score(X_sample, labels, sample_size=5_000, random_state=42)


In [26]:
X_test_honest_scaled = scaler.transform(df_test_honest[feature_cols]).astype(np.float32)
X_test_honest_pca = pca.transform(X_test_honest_scaled)
test_labels_honest = km_final.predict(X_test_honest_pca)
print("silhouette score for test honest split", silhouette_score(X_test_honest_pca, test_labels_honest, sample_size=5_000, random_state=42))


position_raw_th = np.expm1(df_test_honest['gsc_avg_position'].values)
impressions_raw_th = np.expm1(df_test_honest['gsc_impressions'].values)
trend_raw_th = np.sign(df_test_honest['trend_direction'].values) * np.expm1(np.abs(df_test_honest['trend_direction'].values))

nan_mask_th = pd.isna(position_raw_th)
low_impressions_mask_th = (~nan_mask_th) & (impressions_raw_th < 10)
low_conf_mask_th = (~nan_mask_th) & (position_raw_th > 100)
healthy_mask_th = (~nan_mask_th) & (position_raw_th <= 10) & (~low_impressions_mask_th)
declining_mask_th = (~nan_mask_th) & (position_raw_th > 10) & (position_raw_th <= 100) & (trend_raw_th > 0) & (~low_impressions_mask_th)

reason_code_th = np.full(len(df_test_honest), 'WEAK_BUT_STABLE', dtype=object)
reason_code_th[nan_mask_th] = 'NO_DATA'
reason_code_th[low_impressions_mask_th] = 'LOW_CONFIDENCE_SIGNAL'
reason_code_th[low_conf_mask_th] = 'LOW_CONFIDENCE_SIGNAL'
reason_code_th[healthy_mask_th] = 'HEALTHY'
reason_code_th[declining_mask_th] = 'DECLINING_UNDERPERFORMER'

df_test_honest = df_test_honest.copy()
df_test_honest['baseline_reason_code'] = reason_code_th
df_test_honest['cluster'] = test_labels_honest

crosstab_honest = pd.crosstab(df_test_honest['cluster'], df_test_honest['baseline_reason_code'], normalize='index')
print(crosstab_honest.round(3))
print(df_test_honest['cluster'].value_counts())

silhouette score for test honest split 0.7063694
baseline_reason_code  DECLINING_UNDERPERFORMER  ...  WEAK_BUT_STABLE
cluster                                         ...                 
0                                        0.147  ...            0.104
1                                        0.140  ...            0.103
2                                        0.000  ...            0.000

[3 rows x 4 columns]
cluster
0    35262
1     5485
2     5360
Name: count, dtype: int64


In [27]:
priority_map = {
    'DECLINING_UNDERPERFORMER': 1,
    'HEALTHY': 3,
    'WEAK_BUT_STABLE': 2,
    'LOW_CONFIDENCE_SIGNAL': 4,
    'NO_DATA': 5,
}

reason_text_map = {
    'DECLINING_UNDERPERFORMER': 'Has visibility and is losing ground. Refresh candidate.',
    'WEAK_BUT_STABLE': 'Some traction, not clearly moving either way. Monitor or lightly expand.',
    'HEALTHY': 'Performing as expected. No action needed.',
    'LOW_CONFIDENCE_SIGNAL': 'Too little traffic to trust a trend read yet.',
    'NO_DATA': 'No usable position data for this page.',
}

queue = df_test_honest.copy()
queue['priority'] = queue['baseline_reason_code'].map(priority_map)
queue['reason'] = queue['baseline_reason_code'].map(reason_text_map)
queue['cluster_note'] = np.where(
    (queue['cluster'] == 0) & (queue['baseline_reason_code'] == 'LOW_CONFIDENCE_SIGNAL'),
    'Cluster and baseline agree, treat with normal confidence.',
    'Cluster and baseline partially disagree, review before acting.'
)

queue = queue.sort_values('priority')[['content_hash_id', 'priority', 'baseline_reason_code', 'cluster', 'reason', 'cluster_note']]
queue.head(20)

,content_hash_id,priority,baseline_reason_code,cluster,reason,cluster_note
36904,content_000184dde41afe75,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline partially disagree, revie..."
25726,content_00d5be54eede82d6,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline partially disagree, revie..."
131973,content_ff788940775209ac,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline partially disagree, revie..."
75770,content_000184dde41afe75,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline partially disagree, revie..."
97864,content_000184dde41afe75,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline partially disagree, revie..."
112866,content_000184dde41afe75,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline partially disagree, revie..."
81429,content_5a47f99853a015e4,1,DECLINING_UNDERPERFORMER,1,Has visibility and is losing ground. Refresh c...,"Cluster and baseline partially disagree, revie..."
123784,content_5a47f99853a015e4,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline partially disagree, revie..."
138467,content_5a47f99853a015e4,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline partially disagree, revie..."
27392,content_e25e475f477f4187,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline partially disagree, revie..."


In [28]:
# collapse to one row per page, keep the most recent report_date's info
queue_latest = df_test_honest.sort_values('report_date').groupby('content_hash_id').tail(1).copy()

queue_latest['priority'] = queue_latest['baseline_reason_code'].map(priority_map)
queue_latest['reason'] = queue_latest['baseline_reason_code'].map(reason_text_map)

# agreement check: does this row's baseline code match what its cluster usually contains?
dominant_code_per_cluster = crosstab_honest.idxmax(axis=1)
queue_latest['cluster_dominant_code'] = queue_latest['cluster'].map(dominant_code_per_cluster)
queue_latest['cluster_note'] = np.where(
    queue_latest['baseline_reason_code'] == queue_latest['cluster_dominant_code'],
    'Cluster and baseline agree, treat with normal confidence.',
    'Cluster and baseline disagree, review before acting.'
)

queue = queue_latest.sort_values('priority')[
    ['content_hash_id', 'priority', 'baseline_reason_code', 'cluster', 'reason', 'cluster_note']
]
print(f"{len(queue)} unique pages in queue, down from {len(df_test_honest)} page-day rows")
queue.head(20)

2355 unique pages in queue, down from 46107 page-day rows


,content_hash_id,priority,baseline_reason_code,cluster,reason,cluster_note
126266,content_9f76959a4750cbc9,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a..."
130569,content_9e37b416108055a5,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a..."
125273,content_dc9cfc35474f2236,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a..."
142160,content_9f0c23d1ba5745dc,1,DECLINING_UNDERPERFORMER,1,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a..."
124867,content_6ba1851c0f9fe553,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a..."
140657,content_4e8508a5b4860b29,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a..."
142531,content_a2f0e95a8cd314e5,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a..."
130563,content_7db87b7e23fe5da2,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a..."
130620,content_1e2392fe81400941,1,DECLINING_UNDERPERFORMER,0,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a..."
124970,content_3350e4203da3ab2a,1,DECLINING_UNDERPERFORMER,1,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a..."


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.